In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

np.random.seed(42)
n = 50000
base_date = datetime.today()

df = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "dpd": np.random.choice([0, 1, 2, 5, 10, 15, 30, 60, 90], size=n),
    "loan_amount": np.random.randint(5000, 500000, size=n),
    "communication_score": np.random.rand(n),
    "utility_payment_score": np.random.rand(n),
    "sms_response_rate": np.random.rand(n),
    "credit_bureau_score": np.random.randint(300, 900, size=n),
    "last_payment_days_ago": np.random.randint(0, 90, size=n),
    "channel_preference": np.random.choice(["SMS", "WhatsApp", "Call", "Email"], size=n),
    "preferred_hour": np.random.randint(8, 20, size=n),

    # New MSME-related features
    "business_vintage_years": np.random.randint(1, 25, size=n),
    "annual_turnover": np.random.randint(100000, 10000000, size=n),
    "emi_amount": np.random.randint(2000, 50000, size=n),
    "last_repayment_date": [base_date - timedelta(days=int(x)) for x in np.random.randint(1, 90, size=n)],
    "overdue_amount": np.random.randint(0, 200000, size=n),

    # Labels
    "repaid_in_7_days": np.random.binomial(1, 0.3, n),
    "repaid_in_14_days": np.random.binomial(1, 0.4, n),
    "repaid_in_30_days": np.random.binomial(1, 0.6, n),
    "default_risk": np.random.binomial(1, 0.2, n)
})

# Save to CSV
df.to_csv("msme_collections_data.csv", index=False)
print("✅ Dataset generated and saved as 'msme_collections_data.csv'")


✅ Dataset generated and saved as 'msme_collections_data.csv'


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv("msme_collections_data.csv")

features = [
    "dpd", "loan_amount", "communication_score",
    "utility_payment_score", "sms_response_rate", "credit_bureau_score", "business_vintage_years", "annual_turnover",
    "emi_amount", "overdue_amount"
]

df["days_since_last_repayment"] = (base_date - pd.to_datetime(df["last_repayment_date"])).dt.days
features.append("days_since_last_repayment")

X = df[features]

# 1. Repayment probability in 7/14/30 days
y_7 = df["repaid_in_7_days"]
y_14 = df["repaid_in_14_days"]
y_30 = df["repaid_in_30_days"]

X_train, X_test, y7_train, y7_test = train_test_split(X, y_7, test_size=0.2)
_, _, y14_train, y14_test = train_test_split(X, y_14, test_size=0.2)
_, _, y30_train, y30_test = train_test_split(X, y_30, test_size=0.2)

clf7 = RandomForestClassifier().fit(X_train, y7_train)
clf14 = RandomForestClassifier().fit(X_train, y14_train)
clf30 = RandomForestClassifier().fit(X_train, y30_train)

print("7-day:\n", classification_report(y7_test, clf7.predict(X_test)))
print("14-day:\n", classification_report(y14_test, clf14.predict(X_test)))
print("30-day:\n", classification_report(y30_test, clf30.predict(X_test)))

# 2. Channel and Time prediction
from sklearn.preprocessing import LabelEncoder
df["channel_enc"] = LabelEncoder().fit_transform(df["channel_preference"])

clf_channel = RandomForestClassifier().fit(X, df["channel_enc"])
clf_time = RandomForestClassifier().fit(X, df["preferred_hour"])

# 3. Default Risk
clf_risk = RandomForestClassifier().fit(X_train, df.loc[X_train.index, "default_risk"])
print("Default Risk:\n", classification_report(df.loc[X_test.index, "default_risk"], clf_risk.predict(X_test)))

# Save models
joblib.dump(clf7, "clf7.pkl")
joblib.dump(clf14, "clf14.pkl")
joblib.dump(clf30, "clf30.pkl")
joblib.dump(clf_channel, "clf_channel.pkl")
joblib.dump(clf_time, "clf_time.pkl")
joblib.dump(clf_risk, "clf_risk.pkl")


7-day:
               precision    recall  f1-score   support

           0       0.71      1.00      0.83      7067
           1       0.33      0.00      0.01      2933

    accuracy                           0.71     10000
   macro avg       0.52      0.50      0.42     10000
weighted avg       0.59      0.71      0.59     10000

14-day:
               precision    recall  f1-score   support

           0       0.60      0.93      0.73      6030
           1       0.40      0.07      0.12      3970

    accuracy                           0.59     10000
   macro avg       0.50      0.50      0.42     10000
weighted avg       0.52      0.59      0.49     10000

30-day:
               precision    recall  f1-score   support

           0       0.41      0.10      0.16      4072
           1       0.59      0.90      0.72      5928

    accuracy                           0.58     10000
   macro avg       0.50      0.50      0.44     10000
weighted avg       0.52      0.58      0.49     

['clf_risk.pkl']